# Customer Churn Prediction
## Notebook 2: Feature Engineering

### Objective
Prepare customer data for machine learning by separating the target variable, identifying feature types, encoding categorical variables, and creating a stratified train-test split.

### Workflow
1. Load the cleaned dataset.
2. Separate features and target.
3. Identify numerical and categorical features.
4. Split the data into training and test sets.
5. Set up preprocessing while preventing data leakage.

In [12]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = Path("..")
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

df = pd.read_csv(
    PROCESSED_DATA_DIR / "customer_churn_cleaned.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (7032, 21)


## 1. Feature and Target Separation

The target variable is `Churn`, which indicates whether a customer left the service.

The `customerID` column is excluded because it identifies customers but does not provide a meaningful predictive signal by itself.

The target is encoded as 1 for churn and 0 for non-churn.

In [13]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features shape: (7032, 19)
Target shape: (7032,)

Target distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


## 2. Identify Numerical and Categorical Features

Numerical features contain measurable quantities, while categorical features represent groups or labels.

Identifying these types allows us to apply suitable preprocessing to each group.

In [14]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 3. Train-Test Split

Split the data into 80% training and 20% testing sets.

Stratification preserves the approximate proportion of churned and non-churned customers in both sets. The test set will remain unseen during model training.

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining churn rate:", round(y_train.mean() * 100, 2), "%")
print("Testing churn rate:", round(y_test.mean() * 100, 2), "%")

Training features: (5625, 19)
Testing features: (1407, 19)

Training churn rate: 26.58 %
Testing churn rate: 26.58 %


## 4. Define the Preprocessing Pipeline

- Apply `StandardScaler` to numerical features for Logistic Regression.
- Apply `OneHotEncoder` to categorical features.
- Use a scikit-learn pipeline so transformations are fitted on training data only.

The same preprocessing setup will be used when training and evaluating the models. This helps prevent data leakage.

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline defined successfully.")

Preprocessing pipeline defined successfully.
